# Fase 3: Generalización de Legendre (Multipolos l=1, 2, 3)

Buscamos la ley general: $V(r, \theta, l) \propto \frac{1}{r^{l+1}} P_l(\cos \theta)$

### Mejoras Científicas (Pareto Analysis):
1. **Frente de Pareto:** No solo mostramos el "mejor" modelo, sino toda la frontera de complejidad vs exactitud.
2. **Operadores Completos:** No restringimos los operadores para mantener la "honestidad" del descubrimiento.
3. **Variable L:** El sistema debe aprender cómo cambia la ley con el orden del multipolo.

In [ ]:
# 1. Setup
import os
%cd /content
if os.path.exists('ND2'): !rm -rf ND2
!git clone https://github.com/tsinghua-fib-lab/ND2.git
%cd ND2
!pip install gplearn setproctitle geopandas matplotlib
!mkdir -p weights Multipolos/data/nd2_format Multipolos/results/pareto
!wget -O weights/checkpoint.pth https://github.com/yuzhTHU/ND2/releases/download/checkpoint.pth/checkpoint.pth

In [ ]:
%%writefile search.py
import os, json, time, signal, logging, warnings, inspect, traceback
import numpy as np
from socket import gethostname
from argparse import ArgumentParser
from setproctitle import setproctitle
from ND2.model import NDformer
from ND2.utils import init_logger, AutoGPU, seed_all
from ND2.search import MCTS
from ND2.GDExpr import GDExpr
from ND2.search.reward_solver import RewardSolver

warnings.filterwarnings("ignore", category=RuntimeWarning)
def handler(signum, frame): raise KeyboardInterrupt
signal.signal(signal.SIGINT, handler)
logger = logging.getLogger('ND2.search')

def call_robustly(func, **kwargs):
    sig = inspect.signature(func)
    valid_kwargs = {k: v for k, v in kwargs.items() if k in sig.parameters}
    return func(**valid_kwargs)

def main(args):
    data = json.load(open(args.data, 'r'))
    for k, v in data.items(): data[k] = np.array(v)
    rewarder = RewardSolver(Xv={var: data[var] for var in args.vars}, Xe={var: data[var] for var in args.edge_vars},
                            A=data['A'].astype(int), G=data['G'].astype(int), Y=data[args.target_var])
    ndformer = NDformer(device=args.device)
    ndformer.load(args.ndformer_path, weights_only=False)
    ndformer.eval()
    call_robustly(ndformer.set_data, Xv={var: data[var] for var in args.vars}, Xe={var: data[var] for var in args.edge_vars},
                  A=data['A'].astype(int), G=data['G'].astype(int), Y=data[args.target_var], root_type='node', cache_data_emb=True)
    
    mcts_kwargs = {'rewarder': rewarder, 'ndformer': ndformer, 'vars_node': args.vars, 'vars_edge': args.edge_vars, 'log_per_episode': 10, 'beam_size': 10, 'max_coeff_num': 5}
    if args.binary: mcts_kwargs['binary'] = args.binary
    if args.unary: mcts_kwargs['unary'] = args.unary

    est = MCTS(**mcts_kwargs)
    try: call_robustly(est.fit, root_prefix=['node'], episode_limit=args.episodes, max_episodes=args.episodes, time_limit=args.time_limit)
    except KeyboardInterrupt: logger.info('Search Interrupted.')
    except Exception: logger.error(traceback.format_exc())
    finally:
        # EXTRAER FRENTE DE PARETO
        print("\n--- FRENTE DE PARETO EXPLORADO ---")
        pareto_front = est.Pareto(topk=10, print_on_fly=False)
        results = []
        for model, metric in pareto_front:
            eq_str = GDExpr.prefix2str(model)
            print(f"C:{metric['complexity']} | R2:{metric['R2']:.4f} | Eq: {eq_str}")
            results.append({'model': model, 'eq': eq_str, **metric})
        
        os.makedirs("Multipolos/results/pareto", exist_ok=True)
        json.dump(results, open(f"Multipolos/results/pareto/front_{args.name}.json", "w"), indent=2)

if __name__ == '__main__':
    parser = ArgumentParser()
    parser.add_argument('--device', type=str, default='cuda')
    parser.add_argument('--data', type=str, required=True)
    parser.add_argument('--vars', type=str, nargs='*')
    parser.add_argument('--edge_vars', type=str, nargs='*', default=[])
    parser.add_argument('--target_var', type=str, default='target')
    parser.add_argument('--episodes', type=int, default=150)
    parser.add_argument('--binary', type=str, nargs='*', default=None)
    parser.add_argument('--unary', type=str, nargs='*', default=None)
    parser.add_argument('--ndformer_path', type=str, default='./weights/checkpoint.pth')
    parser.add_argument('--time_limit', type=int, default=None)
    parser.add_argument('--name', type=str, default='Search')
    args, _ = parser.parse_known_args()
    seed_all(42)
    init_logger(args.name, f'./log/search/info.log', root_name='ND2')
    main(args)

In [ ]:
# 2. Generador Multi-Multipolo (l=1, 2, 3)
import numpy as np, json, os
from scipy.special import eval_legendre

def generate_multi_l_data(num_samples_per_l=400):
    r_all, theta_all, l_all, V_all = [], [], [], []
    for l_val in [1, 2, 3]: # Dipolo, Cuadrupolo, Octupolo
        r = np.random.uniform(1.0, 5.0, num_samples_per_l)
        theta = np.random.uniform(0, np.pi, num_samples_per_l)
        V = eval_legendre(l_val, np.cos(theta)) / (r**(l_val + 1))
        r_all.extend(r); theta_all.extend(theta); l_all.extend([float(l_val)] * num_samples_per_l); V_all.extend(V)
    
    data = {"V": 2, "E": 1, "A": [[0,1],[0,0]], "G": [[0,1]],
            "l_in": [[float(val), 0.0] for val in l_all],
            "theta": [[0.0, float(t)] for t in theta_all],
            "r_node": [[0.0, float(ri)] for ri in r_all],
            "target": [[0.0, float(vi)] for vi in V_all]}
    os.makedirs("Multipolos/data/nd2_format", exist_ok=True)
    json.dump(data, open("Multipolos/data/nd2_format/LEGENDRE_MULTI_nd2.json", "w"))
    print("Dataset Multi-Multipolo generado.")

generate_multi_l_data()

In [ ]:
# 3. Visualizador del Frente de Pareto (Opcional después de correr search)
def plot_pareto(json_path):
    import matplotlib.pyplot as plt
    import json
    data = json.load(open(json_path))
    comp = [d['complexity'] for d in data]
    r2 = [d['R2'] for d in data]
    plt.figure(figsize=(8,5))
    plt.scatter(comp, r2, color='red')
    plt.plot(comp, r2, linestyle='--')
    plt.xlabel("Complexity"); plt.ylabel("R2 Score")
    plt.title("Frente de Pareto: Complejidad vs Exactitud")
    plt.grid(True); plt.show()

In [ ]:
# 4. Ejecución de Búsqueda (Sin restricciones de operadores)
!python3 search.py \
    --data Multipolos/data/nd2_format/LEGENDRE_MULTI_nd2.json \
    --vars l_in theta r_node \
    --target_var target \
    --episodes 300 \
    --device cuda